# PAD model — analysis

Everything a single AUC hides. Run `pad_model.ipynb` first: this notebook reads
`pad_model_dataset.csv` and `artifacts/model.joblib`, and reproduces the same
splits from the same seeds.

1. Uncertainty — bootstrap confidence interval
2. Subgroup performance — who the model serves worst
3. Feature-group ablation — which groups carry the signal
4. Baseline comparison — do the clinical features beat age and gender?
5. Threshold analysis — the tradeoff AUC abstracts away

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from pad import analysis, calibrate, train

final_model_df = pd.read_csv("pad_model_dataset.csv")
bundle, card = train.load_artifacts()

X, y, groups = train.split_xy(final_model_df)
X_train, X_test, y_train, y_test, groups_train = train.patient_split(X, y, groups)
X_fit, X_cal, y_fit, y_cal, groups_fit = calibrate.split_calibration(
    X_train, y_train, groups_train
)

pipeline = bundle["pipeline"]
y_prob = pipeline.predict_proba(X_test)[:, 1]
y_pred = pipeline.predict(X_test)

print(f"Model: {bundle['model_name']}  |  test rows: {len(y_test)}")
print(f"Cohort: {card.get('cohort')}")

## 1. How uncertain is that AUC?

A point estimate on one split says nothing about its own stability. The
interval below is what should be quoted, not the point estimate alone.

In [ ]:
ci = analysis.bootstrap_auc(y_test, y_prob)
print(f"Test AUC {ci['auc']:.4f}   95% CI [{ci['lo']:.4f}, {ci['hi']:.4f}]"
      f"   ({ci['n_resamples']} resamples)")
print(f"Interval width: {ci['hi'] - ci['lo']:.4f}")

analysis.classification_summary(y_test, y_pred)

## 2. Subgroup performance

An aggregate number can hide a subgroup the model fails. Group sizes are shown
alongside: a subgroup too small to estimate is reported as such rather than
given a number that looks like evidence.

In [ ]:
subgroups = analysis.subgroup_performance(X_test, y_test, y_prob)
subgroups

In [ ]:
estimable = subgroups.dropna(subset=["auc"])

if len(estimable):
    fig, ax = plt.subplots(figsize=(7, 4))
    labels = estimable["group"] + ": " + estimable["subgroup"]
    ax.barh(labels, estimable["auc"], color="steelblue")
    ax.axvline(ci["auc"], color="black", ls="--", lw=1, label="overall AUC")
    ax.set_xlim(0.5, 1.0)
    ax.set_xlabel("AUC")
    ax.set_title("Performance by subgroup")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No subgroup had enough rows to estimate an AUC.")

## 3. Feature-group ablation

Each group dropped in turn, scored by cross-validation on the training split.
This is a model development question, so it is answered without touching the
test set.

In [ ]:
ablation = analysis.ablation(X_fit, y_fit, groups_fit)
ablation

## 4. Does the clinical data earn its place?

Age and gender alone, against the full feature set. If the gap is small, the
honest conclusion is that the labs and history are not adding much.

In [ ]:
analysis.demographics_baseline(X_fit, y_fit, groups_fit, X_test, y_test)

## 5. Threshold analysis

The project argues for AUC because the operating threshold is tunable to the
cost of a missed case. This is that tradeoff, made explicit.

In [ ]:
thresholds = analysis.threshold_table(y_test, y_prob)
thresholds

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for column in ["sensitivity", "specificity", "ppv", "npv"]:
    ax.plot(thresholds["threshold"], thresholds[column], marker="o", label=column)
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Rate")
ax.set_title("Operating characteristics across thresholds")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Caveat

These numbers describe whatever data `pad_model.ipynb` was last run on. If that
was the synthetic fixture rather than real MIMIC-IV, they are a smoke test of
the code and nothing more — check `data_provenance` in the model card.

In [ ]:
card.get("data_provenance", "not recorded — assume real MIMIC-IV")